**Uwaga** W poniższych zadaniach zakładamy, iż serwer powinien obsługiwać tylko jednego klienta w danej chwili.

In [ ]:
import socket
import imaplib
import base64
from email.parser import BytesParser

In [ ]:
HOST = "127.0.0.1"
PORT = 9143

def connect_imap():
    imap = imaplib.IMAP4(HOST, PORT)
    return imap

def login_imap(imap, username, password):
    try:
        imap.login(username, password)
        print(f"Successfully logged in as {username}")
        return True
    except imaplib.IMAP4.error as e:
        print(f"Login failed: {e}")
        return False

def list_mailboxes(imap):
    status, mailboxes = imap.list()
    if status == 'OK':
        print("Available mailboxes:")
        mailbox_names = []
        for mailbox_info in mailboxes:
            mailbox_name = mailbox_info.decode().split('"')[-2]
            print(f"  - {mailbox_name}")
            mailbox_names.append(mailbox_name)
        return mailbox_names
    return []

def count_messages(imap, mailbox_name):
    status, msg_count = imap.select(mailbox_name)
    if status == 'OK':
        return int(msg_count[0])
    return 0

def get_unseen_messages(imap):
    status, msg_list = imap.search(None, 'UNSEEN')
    if status == 'OK':
        return msg_list[0].split()
    return []

def get_all_messages(imap):
    status, msg_list = imap.search(None, 'ALL')
    if status == 'OK':
        return msg_list[0].split()
    return []

def disconnect_imap(imap):
    imap.close()
    imap.logout()

1. Wykorzystując protokół telnet oraz serwer IMAP, zaloguj się do skrzynki i sprawdź, ile wiadomości znajduje się w poszczególnych skrzynkach. Pobierz pierwszą dostępną wiadomość i oznacz ją jako przeczytaną.
Wykorzystaj komendę protokołu IMAP - STORE.

In [ ]:
def ex01():
    print("=== IMAP Client - Zadanie 1 ===")
    print()

    try:
        imap = connect_imap()

        if not login_imap(imap, "test@example.com", "password"):
            imap.logout()
            return

        print("\n--- Mailbox Statistics ---")
        mailbox_names = list_mailboxes(imap)

        for mailbox_name in mailbox_names:
            count = count_messages(imap, mailbox_name)
            print(f"{mailbox_name}: {count} wiadomości")

        print("\n--- Pobieranie i oznaczenie pierwszej wiadomości ---")
        imap.select('INBOX')
        status, msg_list = imap.search(None, 'ALL')

        if status == 'OK' and msg_list[0]:
            mail_ids = msg_list[0].split()
            first_mail_id = mail_ids[0]

            status, msg_data = imap.fetch(first_mail_id, '(RFC822)')
            if status == 'OK':
                print(f"Pobrana wiadomość ID: {first_mail_id.decode()}")
                print(msg_data[0][1].decode()[:300])

            imap.store(first_mail_id, '+FLAGS', '\\Seen')
            print(f"\nWiadomość ID {first_mail_id.decode()} oznaczona jako przeczytana")
        else:
            print("Brak wiadomości w INBOX")

        disconnect_imap(imap)
        print("\n✓ Zadanie 1 ukończone")

    except Exception as e:
        print(f"Błąd: {e}")

In [ ]:
ex01()

2. Napisz program klienta, który połączy się z serwerem IMAP, a następnie wyświetli informację o tym, ile wiadomości znajduje się w skrzynce Inbox.

In [ ]:
def ex02():
    print("=== IMAP Client - Zadanie 2 ===")
    print()

    try:
        imap = connect_imap()

        if not login_imap(imap, "test@example.com", "password"):
            imap.logout()
            return

        inbox_count = count_messages(imap, 'INBOX')
        print(f"\nLiczba wiadomości w INBOX: {inbox_count}")

        disconnect_imap(imap)
        print("\n✓ Zadanie 2 ukończone")

    except Exception as e:
        print(f"Błąd: {e}")

In [ ]:
ex02()

3. Napisz program klienta, który połączy się z serwerem IMAP, a następnie wyświetli informację o tym, ile wiadomości znajduje się we wszystkich skrzynkach łącznie.

In [ ]:
def ex03():
    print("=== IMAP Client - Zadanie 3 ===")
    print()

    try:
        imap = connect_imap()

        if not login_imap(imap, "test@example.com", "password"):
            imap.logout()
            return

        print("\n--- Liczba wiadomości w poszczególnych skrzynkach ---")
        mailbox_names = list_mailboxes(imap)

        total_messages = 0
        for mailbox_name in mailbox_names:
            count = count_messages(imap, mailbox_name)
            print(f"{mailbox_name}: {count}")
            total_messages += count

        print(f"\nŁączna liczba wiadomości we wszystkich skrzynkach: {total_messages}")

        disconnect_imap(imap)
        print("\n✓ Zadanie 3 ukończone")

    except Exception as e:
        print(f"Błąd: {e}")

In [ ]:
ex03()

4. Napisz program klienta, który połączy się z serwerem IMAP, a następnie sprawdzi, czy w skrzynce są nieprzeczytane wiadomości. Jeśli tak, wyświetli treść wszystkich nieprzeczytanych wiadomości oraz oznaczy je jako przeczytane (komenda STORE i flagi - FLAGS).

In [ ]:
def ex04():
    print("=== IMAP Client - Zadanie 4 ===")
    print()

    try:
        imap = connect_imap()

        if not login_imap(imap, "test@example.com", "password"):
            imap.logout()
            return

        imap.select('INBOX')
        unseen_ids = get_unseen_messages(imap)

        if unseen_ids:
            print(f"\nZnaleziono {len(unseen_ids)} nieprzeczytanych wiadomości:\n")

            for mail_id in unseen_ids:
                status, msg_data = imap.fetch(mail_id, '(RFC822)')
                if status == 'OK':
                    email_body = msg_data[0][1].decode(errors='replace')
                    print(f"--- Wiadomość ID: {mail_id.decode()} ---")
                    print(email_body[:400])
                    print()

                    imap.store(mail_id, '+FLAGS', '\\Seen')

            print("✓ Wszystkie nieprzeczytane wiadomości zostały oznaczone jako przeczytane")
        else:
            print("\nBrak nieprzeczytanych wiadomości w INBOX")

        disconnect_imap(imap)
        print("\n✓ Zadanie 4 ukończone")

    except Exception as e:
        print(f"Błąd: {e}")

In [ ]:
ex04()

5. Napisz program klienta, który połączy się z serwerem IMAP, a następnie fizycznie usunie wybraną wiadomość.

In [ ]:
def ex05():
    print("=== IMAP Client - Zadanie 5 ===")
    print()

    try:
        imap = connect_imap()

        if not login_imap(imap, "test@example.com", "password"):
            imap.logout()
            return

        imap.select('INBOX')

        all_ids = get_all_messages(imap)

        if all_ids:
            print(f"\nWiadomości w INBOX: {len(all_ids)}")

            mail_to_delete = all_ids[0]
            print(f"Usuwanie wiadomości o ID: {mail_to_delete.decode()}")

            imap.store(mail_to_delete, '+FLAGS', '\\Deleted')
            imap.expunge()

            print("✓ Wiadomość została usunięta")

            remaining_ids = get_all_messages(imap)
            print(f"\nPozostało wiadomości w INBOX: {len(remaining_ids)}")
        else:
            print("\nBrak wiadomości do usunięcia")

        disconnect_imap(imap)
        print("\n✓ Zadanie 5 ukończone")

    except Exception as e:
        print(f"Błąd: {e}")

In [ ]:
ex05()

6. Napisz program serwera, który działając pod adresem 127.0.0.1 oraz na określonym porcie TCP, będzie serwerem poczty, obsługującym protokół IMAP. Nie realizuj faktycznego pobierania e-maili, tylko zasymuluj jego działanie tak, żeby napisany wcześniej klient IMAP mógł pobrac wiadomosci. Pamiętaj o obsłudze przypadku, gdy klient poda nie zaimplementowaną przez serwer komendę.

Zadanie 6 wykonane w pliku ```server_zad06.py```